In [2]:
import mlflow
import mlflow.sklearn

import json
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("MLflow:", mlflow.__version__)
print("Phase 4 libraries imported successfully.")

MLflow: 3.16.1
Phase 4 libraries imported successfully.


In [3]:
PROJECT_ROOT = Path("..").resolve()

MODEL_DIR = PROJECT_ROOT / "models"

MODEL_PATH = MODEL_DIR / "candidate_model.joblib"
METADATA_PATH = MODEL_DIR / "candidate_metadata.json"

MLFLOW_DB = PROJECT_ROOT / "mlflow.db"

print("Project root :", PROJECT_ROOT)
print("Model path   :", MODEL_PATH)
print("Metadata path:", METADATA_PATH)
print("MLflow DB    :", MLFLOW_DB)

Project root : C:\Users\Menaka\Videos\smart-food-demand-mlops
Model path   : C:\Users\Menaka\Videos\smart-food-demand-mlops\models\candidate_model.joblib
Metadata path: C:\Users\Menaka\Videos\smart-food-demand-mlops\models\candidate_metadata.json
MLflow DB    : C:\Users\Menaka\Videos\smart-food-demand-mlops\mlflow.db


In [4]:
print("=" * 70)
print("PHASE 3 ARTIFACT VERIFICATION")
print("=" * 70)

print("Model exists   :", MODEL_PATH.exists())
print("Metadata exists:", METADATA_PATH.exists())

PHASE 3 ARTIFACT VERIFICATION
Model exists   : True
Metadata exists: True


In [5]:
tracking_uri = f"sqlite:///{MLFLOW_DB.as_posix()}"

mlflow.set_tracking_uri(tracking_uri)

print("MLflow tracking URI:")
print(mlflow.get_tracking_uri())

MLflow tracking URI:
sqlite:///C:/Users/Menaka/Videos/smart-food-demand-mlops/mlflow.db


In [6]:
EXPERIMENT_NAME = "Smart Food Demand Forecasting"

mlflow.set_experiment(EXPERIMENT_NAME)

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

print("=" * 70)
print("MLFLOW EXPERIMENT")
print("=" * 70)

print("Experiment name:", experiment.name)
print("Experiment ID  :", experiment.experiment_id)

2026/09/23 21:45:59 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/23 21:45:59 INFO mlflow.store.db.utils: Updating database tables
2026/09/23 21:46:01 INFO mlflow.tracking.fluent: Experiment with name 'Smart Food Demand Forecasting' does not exist. Creating a new experiment.


MLFLOW EXPERIMENT
Experiment name: Smart Food Demand Forecasting
Experiment ID  : 1


In [7]:
with open(METADATA_PATH, "r") as f:
    model_metadata = json.load(f)

print("=" * 70)
print("MODEL METADATA")
print("=" * 70)

print("Model :", model_metadata["model_name"])
print("Target:", model_metadata["target"])

print("\nMetrics:")
print("MAE :", model_metadata["mae"])
print("RMSE:", model_metadata["rmse"])
print("R²  :", model_metadata["r2"])

MODEL METADATA
Model : Linear Regression
Target: sales

Metrics:
MAE : 0.2685755437789722
RMSE: 0.3475217130796383
R²  : 0.8001205800725606


In [8]:
import joblib

candidate_model = joblib.load(MODEL_PATH)

print("=" * 70)
print("MODEL LOADED")
print("=" * 70)

print("Model type:", type(candidate_model))

MODEL LOADED
Model type: <class 'sklearn.pipeline.Pipeline'>


In [9]:
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"

train_df = pd.read_csv(TRAIN_PATH)

train_df["date"] = pd.to_datetime(train_df["date"])


def create_date_features(df):
    df = df.copy()

    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["day_of_week"] = df["date"].dt.dayofweek
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    df["quarter"] = df["date"].dt.quarter
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    return df


train_df = create_date_features(train_df)

model_df = (
    train_df
    .sort_values(["store", "date"])
    .reset_index(drop=True)
)

model_df["sales_lag_1"] = (
    model_df
    .groupby("store")["sales"]
    .shift(1)
)

model_df["sales_lag_7"] = (
    model_df
    .groupby("store")["sales"]
    .shift(7)
)

model_df["sales_rolling_mean_7"] = (
    model_df
    .groupby("store")["sales"]
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

lag_features = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_rolling_mean_7"
]

model_df = (
    model_df
    .dropna(subset=lag_features)
    .sort_values(["date", "store"])
    .reset_index(drop=True)
)

unique_dates = np.sort(model_df["date"].unique())

split_index = int(len(unique_dates) * 0.80)

split_date = unique_dates[split_index]

train_model = model_df[
    model_df["date"] < split_date
].copy()

validation_model = model_df[
    model_df["date"] >= split_date
].copy()

print("Validation shape:", validation_model.shape)
print("Validation start:", validation_model["date"].min())
print("Validation end  :", validation_model["date"].max())

Validation shape: (1397, 23)
Validation start: 2023-06-16 00:00:00
Validation end  : 2023-11-30 00:00:00


In [10]:
TARGET = "sales"

MODEL_FEATURES = [
    "store",
    "is_state_holiday",
    "is_school_holiday",
    "is_special_day",
    "temperature_max",
    "temperature_min",
    "temperature_mean",
    "sunshine_sum",
    "precipitation_sum",
    "year",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "quarter",
    "is_weekend",
    "sales_lag_1",
    "sales_lag_7",
    "sales_rolling_mean_7"
]

X_validation = validation_model[MODEL_FEATURES]
y_validation = validation_model[TARGET]

print("Validation features:", X_validation.shape)
print("Validation target   :", y_validation.shape)

Validation features: (1397, 19)
Validation target   : (1397,)


In [11]:
predictions = candidate_model.predict(X_validation)

mae = mean_absolute_error(
    y_validation,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        predictions
    )
)

r2 = r2_score(
    y_validation,
    predictions
)

print("=" * 70)
print("VALIDATION METRICS")
print("=" * 70)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

VALIDATION METRICS
MAE  : 0.2686
RMSE : 0.3475
R²   : 0.8001


In [12]:
RUN_NAME = "Linear Regression - Phase 3 Candidate"

with mlflow.start_run(run_name=RUN_NAME) as run:

    run_id = run.info.run_id

    print("=" * 70)
    print("MLFLOW RUN CREATED")
    print("=" * 70)

    print("Run ID:", run_id)

MLFLOW RUN CREATED
Run ID: 22cf9a56f08148589eb619da9f340c98


In [14]:
REGISTERED_MODEL_NAME = "SmartFoodDemandModel"

with mlflow.start_run(
    run_name="Linear Regression - Phase 3 Candidate"
) as run:

    # --------------------------------------------------
    # Parameters
    # --------------------------------------------------

    mlflow.log_param(
        "model_type",
        "Linear Regression"
    )

    mlflow.log_param(
        "target",
        TARGET
    )

    mlflow.log_param(
        "num_features",
        len(MODEL_FEATURES)
    )

    mlflow.log_param(
        "training_rows",
        len(train_model)
    )

    mlflow.log_param(
        "validation_rows",
        len(validation_model)
    )

    mlflow.log_param(
        "split_date",
        str(split_date)
    )

    # --------------------------------------------------
    # Metrics
    # --------------------------------------------------

    mlflow.log_metric("mae", float(mae))
    mlflow.log_metric("rmse", float(rmse))
    mlflow.log_metric("r2", float(r2))

    # Baseline metrics
    mlflow.log_metric(
        "baseline_mae",
        float(model_metadata["baseline_mae"])
    )

    mlflow.log_metric(
        "baseline_rmse",
        float(model_metadata["baseline_rmse"])
    )

    mlflow.log_metric(
        "baseline_r2",
        float(model_metadata["baseline_r2"])
    )

    # --------------------------------------------------
    # Tags
    # --------------------------------------------------

    mlflow.set_tag(
        "project",
        "Smart Food Demand Forecasting"
    )

    mlflow.set_tag(
        "phase",
        "Phase 4 - MLflow Model Management"
    )

    mlflow.set_tag(
        "candidate_status",
        "candidate"
    )

    mlflow.set_tag(
        "validation_type",
        "chronological"
    )

    # --------------------------------------------------
    # Artifacts
    # --------------------------------------------------

    mlflow.log_artifact(
        str(METADATA_PATH),
        artifact_path="metadata"
    )

    # --------------------------------------------------
    # Log and register model
    # --------------------------------------------------

    mlflow.sklearn.log_model(
        sk_model=candidate_model,
        name="model",
        registered_model_name=REGISTERED_MODEL_NAME,
        skops_trusted_types=["numpy.dtype"]
    )

    print("=" * 70)
    print("MLFLOW RUN COMPLETED")
    print("=" * 70)

    print("Run ID:", run.info.run_id)
    print("Registered model:", REGISTERED_MODEL_NAME)

MLFLOW RUN COMPLETED
Run ID: 82dd71e54f0f4ce886cb1b37174cc6e5
Registered model: SmartFoodDemandModel


Successfully registered model 'SmartFoodDemandModel'.
Created version '1' of model 'SmartFoodDemandModel'.


In [15]:
client = mlflow.MlflowClient()

experiment = client.get_experiment_by_name(
    EXPERIMENT_NAME
)

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"]
)

latest_run = runs[0]

print("=" * 70)
print("LATEST MLFLOW RUN")
print("=" * 70)

print("Run ID:", latest_run.info.run_id)

print("\nParameters:")
for key, value in latest_run.data.params.items():
    print(f"{key}: {value}")

print("\nMetrics:")
for key, value in latest_run.data.metrics.items():
    print(f"{key}: {value}")

LATEST MLFLOW RUN
Run ID: 82dd71e54f0f4ce886cb1b37174cc6e5

Parameters:
model_type: Linear Regression
target: sales
num_features: 19
training_rows: 3682
validation_rows: 1397
split_date: 2023-06-16T00:00:00.000000

Metrics:
mae: 0.2685755437789722
rmse: 0.3475217130796383
r2: 0.8001205800725606
baseline_mae: 0.2592931952471491
baseline_rmse: 0.3829885886986439
baseline_r2: 0.7572407092557709


In [16]:
registered_versions = client.search_model_versions(
    f"name='{REGISTERED_MODEL_NAME}'"
)

print("=" * 70)
print("REGISTERED MODEL")
print("=" * 70)

for version in registered_versions:
    print("Model name :", version.name)
    print("Version    :", version.version)
    print("Run ID     :", version.run_id)
    print("Status     :", version.status)

REGISTERED MODEL
Model name : SmartFoodDemandModel
Version    : 1
Run ID     : 82dd71e54f0f4ce886cb1b37174cc6e5
Status     : READY


In [17]:
latest_version = max(
    registered_versions,
    key=lambda x: int(x.version)
)

MODEL_VERSION = latest_version.version

model_uri = (
    f"models:/{REGISTERED_MODEL_NAME}/{MODEL_VERSION}"
)

print("=" * 70)
print("REGISTERED MODEL URI")
print("=" * 70)

print("Model name:", REGISTERED_MODEL_NAME)
print("Version   :", MODEL_VERSION)
print("URI       :", model_uri)

REGISTERED MODEL URI
Model name: SmartFoodDemandModel
Version   : 1
URI       : models:/SmartFoodDemandModel/1


In [18]:
registered_model = mlflow.sklearn.load_model(
    model_uri
)

print("=" * 70)
print("REGISTERED MODEL LOADED")
print("=" * 70)

print("Model URI :", model_uri)
print("Model type:", type(registered_model))

REGISTERED MODEL LOADED
Model URI : models:/SmartFoodDemandModel/1
Model type: <class 'sklearn.pipeline.Pipeline'>


In [19]:
registry_predictions = registered_model.predict(
    X_validation
)

registry_mae = mean_absolute_error(
    y_validation,
    registry_predictions
)

registry_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        registry_predictions
    )
)

registry_r2 = r2_score(
    y_validation,
    registry_predictions
)

print("=" * 70)
print("REGISTERED MODEL VERIFICATION")
print("=" * 70)

print(f"MAE  : {registry_mae:.4f}")
print(f"RMSE : {registry_rmse:.4f}")
print(f"R²   : {registry_r2:.4f}")

print(
    "\nPredictions match:",
    np.allclose(
        predictions,
        registry_predictions
    )
)

REGISTERED MODEL VERIFICATION
MAE  : 0.2686
RMSE : 0.3475
R²   : 0.8001

Predictions match: True
